# Projet A6 · Génération de texte : du bigramme au LLM adapté avec LoRA · ⭐⭐⭐

**Piste « Projets avancés » · Niveau ⭐⭐⭐ Avancé** · Pipeline complet façon soutenance : données → nettoyage → indicateurs → modèles → fine-tuning → interprétation → recommandation.

- **Question métier** : une maison d'édition veut un assistant qui écrit *dans le style* d'un auteur du domaine public (Alexandre Dumas) pour des ateliers d'écriture. Quelle approche donne le meilleur rapport qualité / coût : un petit modèle entraîné de zéro, ou un LLM pré-entraîné adapté avec quelques paramètres ?
- **Ce qu'on va construire** : un modèle bigramme (rappel de la séance 9), un **LSTM caractère par caractère** en Keras, puis le **fine-tuning de `Qwen2.5-0.5B-Instruct` avec LoRA** (bibliothèque `peft`), comparés sur les mêmes amorces et avec la **perplexité** mesurée sur un extrait tenu à part.
- **Livrable** : ce notebook exécuté + un rapport de 10-15 slides (gabarit `../gabarit-rapport.md`) + une phrase de synthèse.

**Comment l'utiliser**
- Google Colab avec GPU : menu *Exécution → Modifier le type d'exécution → T4 GPU*. En local, un CPU suffit en mode rapide.
- `MODE_RAPIDE = True` (par défaut) : 100 000 caractères, quelques epochs, 64 exemples pour LoRA → moins de 10 minutes sur CPU. `MODE_RAPIDE = False` : corpus complet, ≈ 20 minutes sur T4.
- Une cellule à la fois, `Maj + Entrée`. Les cellules « À toi » ont un squelette qui s'exécute tel quel et une solution masquée : essaie d'abord.

## 0. Préparation

Imports, flag `MODE_RAPIDE`, helpers `verifier()` / `proche()`, dossier `data/` (ignoré par git) pour le cache des téléchargements.

In [ ]:
# Colab n'a pas PEFT par défaut.
# On n'installe que ce qui manque : la cellule ne fait rien si tout est déjà là.
import importlib.util, subprocess, sys

manquants = [paquet for module, paquet in [("transformers", "transformers"), ("peft", "peft"), ("accelerate", "accelerate")]
             if importlib.util.find_spec(module) is None]
if manquants:
    print("installation :", ", ".join(manquants))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *manquants], check=True)
else:
    print("Tout est déjà installé.")


In [ ]:
import os
import re
import math
import time
import random
import collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODE_RAPIDE = True        # True : < 10 min sur CPU · False : version complète (Colab T4, ≈ 20 min)
GRAINE = 42
random.seed(GRAINE)
np.random.seed(GRAINE)
os.makedirs("data", exist_ok=True)
print("MODE_RAPIDE =", MODE_RAPIDE)

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans arrêter le notebook."""
    print(("✅ " if condition else "❌ ") + nom)

def proche(a, b, tol=0.02):
    """Vrai si a et b sont égaux à tol près (en relatif)."""
    return abs(a - b) <= tol * max(abs(a), abs(b), 1e-9)

TEMPS = {}     # temps d'entraînement de chaque modèle, pour le tableau comparatif

import subprocess, sys
try:
    import peft
except ImportError:                       # Colab n'a pas peft par défaut
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "accelerate"], check=True)
    import peft
import torch
import transformers
import tensorflow as tf

tf.random.set_seed(GRAINE)
torch.manual_seed(GRAINE)
APPAREIL = "cuda" if torch.cuda.is_available() else "cpu"
print("tensorflow", tf.__version__, "· transformers", transformers.__version__, "· peft", peft.__version__)
print("Calcul sur :", APPAREIL)

## 1. Contexte et question métier

**Le problème.** Une maison d'édition anime des ateliers d'écriture autour des classiques. Elle voudrait un outil qui, à partir d'une amorce (« Dantès regarda… »), propose une suite *dans le style de Dumas* : pas pour remplacer l'auteur, mais pour lancer les participants, illustrer un procédé de style, ou fabriquer des exercices « vrai ou faux Dumas ». Le texte est dans le domaine public (Dumas est mort en 1870), donc on peut l'utiliser librement pour entraîner un modèle.

**À qui ça sert.** Aux animateurs (fabriquer du matériel rapidement), aux participants (voir ce qu'une machine « comprend » du style), et à l'équipe technique qui doit choisir une approche : un petit modèle maison, ou un grand modèle pré-entraîné qu'on adapte ? Le coût n'est pas le même : quelques minutes de CPU d'un côté, un GPU et un modèle de 500 millions de paramètres de l'autre.

**Ce qu'un bon résultat veut dire.** Un texte en français correct, cohérent sur quelques phrases, qui *sonne* comme Dumas (dialogues à tirets, « Monsieur », marins, Marseille) sans recopier le livre mot pour mot. Et une **décision** : quelle approche recommander, pour quel coût.

**La métrique choisie.** Juger un texte « à l'œil » est subjectif ; on ajoute donc une mesure : la **perplexité** sur un extrait du livre que le modèle n'a jamais vu. La perplexité, c'est « entre combien de possibilités le modèle hésite » à chaque caractère : 1 = il connaît la suite par cœur, 90 = il tire au hasard parmi les 90 caractères du vocabulaire. Plus elle est basse, mieux le modèle a capté la langue *et* le style. On la calcule **par caractère** pour comparer des modèles qui ne découpent pas le texte de la même façon (caractères pour le LSTM, tokens pour Qwen). Elle a ses limites (elle ne mesure ni la créativité ni la fidélité au style), d'où une grille de lecture qualitative en section 7.

## 2. Les données

Le corpus : *Le Comte de Monte-Cristo*, tome I (1845), Alexandre Dumas, dans l'édition du Projet Gutenberg (livre n° 17989, ≈ 750 000 caractères, licence Project Gutenberg — texte libre). Le fichier est téléchargé une fois et mis en cache dans `data/`. Si le téléchargement échoue, un court texte de secours écrit pour ce notebook prend le relais (tout fonctionne, les résultats sont juste moins bons). Un corpus n'a pas de colonnes comme un tableau, mais on décrit quand même dans un dictionnaire les « variables » qu'on va manipuler.

In [ ]:
TEXTE_SECOURS = """Le port dormait encore quand la barque du vieux pêcheur quitta la jetée. Le ciel, gris comme une lame, ne laissait
deviner ni le jour ni la tempête. Assis à l'arrière, le jeune homme regardait s'éloigner la ville sans dire un mot.
— Vous ne parlez pas, dit le pêcheur.
— Je pense, répondit-il. Je pense à ce que je dirai quand je reviendrai, et à ceux qui ne m'attendent plus.
Le vieil homme haussa les épaules. Il avait vu partir bien des hommes ; peu revenaient avec la même figure.
— La mer ne rend rien, dit-il enfin. Elle garde ce qu'on lui donne et se moque de ce qu'on espère.
Une heure plus tard, ils doublaient la pointe où la vigie, autrefois, signalait les navires venus de Smyrne et de
Naples. Le jeune homme se leva, chercha des yeux la maison blanche où il avait grandi, et ne la trouva pas.
— On l'a vendue, dit le pêcheur sans se retourner. Le notaire est venu avec deux hommes. Votre père n'a rien dit.
— Mon père ne disait jamais rien, murmura-t-il. C'est moi qui parlerai pour lui.
Il se rassit. La barque filait maintenant sous une brise tiède ; les voiles claquaient comme des mains qu'on frappe
l'une contre l'autre pour se réchauffer. Au loin, un trois-mâts remontait vers le port, lent et sûr, la coque noire,
le pavillon en berne. Le pêcheur le désigna du menton.
— Celui-là revient. Il a perdu son capitaine, à ce qu'on dit. Un bon marin, pourtant.
— Les bons marins meurent comme les autres, dit le jeune homme. C'est pour cela que je ne veux pas être marin.
— Et que voulez-vous être ?
— Riche, dit-il. Et patient.
Le pêcheur rit, d'un rire bref qui ressemblait à une toux. Il n'avait jamais rencontré personne qui fût les deux.
Ils abordèrent l'île à midi. Une tour ruinée dominait la crique ; des pierres, roulées par des siècles de vent,
formaient au pied de la falaise une sorte d'escalier que nul n'avait taillé. Le jeune homme sauta sur le sable.
— Attendez-moi jusqu'au soir, dit-il. Si je ne reviens pas, repartez, et ne dites à personne où vous m'avez laissé.
— Et si vous revenez ?
— Alors, dit-il en souriant pour la première fois, vous serez le premier homme de Marseille à qui je ferai du bien.
Il gravit l'escalier de pierre sans se retourner. La tour, de près, paraissait plus haute et plus vieille ; une
ouverture, à hauteur d'homme, laissait voir un couloir sombre où l'air sentait le sel et la cendre. Il y entra.
Il compta vingt pas, puis trente, puis la lumière manqua tout à fait. Il s'arrêta, écouta : le vent, la mer, et,
plus faible, un bruit régulier qui n'était ni l'un ni l'autre. Quelqu'un, quelque part, frappait la pierre.
— Qui est là ? cria-t-il.
Le bruit cessa. Puis une voix, si lasse qu'elle semblait venir du fond des années, répondit :
— Quelqu'un qui attend depuis longtemps. Approchez, monsieur. Nous avons, je crois, beaucoup à nous dire.
"""
print(len(TEXTE_SECOURS), "caractères de secours")

In [ ]:
import requests

URL_DUMAS = "https://www.gutenberg.org/cache/epub/17989/pg17989.txt"
CHEMIN = "data/monte_cristo_t1.txt"

def charger_gutenberg(url, chemin):
    if not os.path.exists(chemin):
        reponse = requests.get(url, timeout=60)
        reponse.raise_for_status()
        with open(chemin, "w", encoding="utf-8") as f:
            f.write(reponse.text)
    with open(chemin, encoding="utf-8") as f:
        return f.read()

try:
    texte_brut = charger_gutenberg(URL_DUMAS, CHEMIN)
    SOURCE = "Gutenberg 17989 · Le Comte de Monte-Cristo, tome I"
except Exception as erreur:
    print("Téléchargement impossible (", erreur, ") → texte de secours intégré")
    texte_brut = TEXTE_SECOURS
    SOURCE = "texte de secours"
print(SOURCE, "·", f"{len(texte_brut):,}".replace(",", " "), "caractères")

**Colle ton propre texte (optionnel).** Tu peux remplacer Dumas par n'importe quel texte dont tu as le droit d'usage : tes propres écrits, un autre livre du domaine public (Gutenberg, Wikisource), des paroles de chansons libres… Il faut au moins 20 000 caractères pour que l'entraînement ait un sens. Si tu changes de corpus, adapte aussi les amorces de la section 5.

In [ ]:
MON_TEXTE = """
"""      # ← colle ton texte entre les guillemets (au moins 20 000 caractères) ; laisse vide pour garder Dumas

if len(MON_TEXTE.strip()) >= 20_000:
    texte_brut = MON_TEXTE
    SOURCE = "mon texte"
    print("Ton texte est utilisé :", len(texte_brut), "caractères")
else:
    print("Corpus utilisé :", SOURCE)

In [ ]:
dictionnaire = pd.DataFrame([
    ["texte_brut",  "str",        "le fichier tel que téléchargé (en-tête et pied Gutenberg compris)", "« The Project Gutenberg eBook of… »"],
    ["texte",       "str",        "le roman nettoyé (section 3), coupé à 100 000 caractères en mode rapide", "« Le 24 février 1815, la vigie… »"],
    ["texte_test",  "str",        "les 5 % finaux de `texte`, jamais vus à l'entraînement (perplexité)", "dernier chapitre disponible"],
    ["caracteres",  "list[str]",  "le vocabulaire des caractères (lettres, accents, ponctuation)", "['\\n', ' ', '!', …, 'û']"],
    ["ids",         "np.ndarray", "le texte encodé caractère par caractère en entiers", "[38, 52, 1, 15, …]"],
    ["mots",        "list[str]",  "le texte découpé en mots (regex), pour l'exploration", "['Le', '24', 'février', …]"],
], columns=["variable", "type", "description", "exemple"])
dictionnaire

In [ ]:
print(texte_brut[:700])
print("…")
print("Caractères :", len(texte_brut), "· lignes :", texte_brut.count("\n"), "· type :", type(texte_brut).__name__)
lignes = texte_brut.split("\n")
print("Lignes vides (« manquants ») :", sum(1 for l in lignes if not l.strip()), "sur", len(lignes))

## 3. Nettoyage et feature engineering

Chaque correction est notée dans un **journal** (liste de dicts) : c'est ce qu'on montre en soutenance pour prouver qu'on sait ce qu'on a fait aux données. Le fichier Gutenberg contient une licence en anglais avant et après le roman, des espaces multiples, des soulignés `_` pour l'italique, et une poignée de caractères très rares qui compliqueraient la vie du modèle caractère par caractère.

In [ ]:
JOURNAL = []

def noter(etape, avant, apres, pourquoi):
    JOURNAL.append({"étape": etape, "caractères avant": avant, "caractères après": apres, "pourquoi": pourquoi})

texte = texte_brut
if "*** START" in texte and "*** END" in texte:
    debut = texte.index("\n", texte.index("*** START"))
    fin = texte.index("*** END")
    avant = len(texte)
    texte = texte[debut:fin]
    noter("retirer l'en-tête et le pied Gutenberg", avant, len(texte), "licence en anglais, hors du roman")
print(len(texte), "caractères après retrait des marqueurs")

In [ ]:
avant = len(texte)
texte = texte.replace("_", "")                       # italiques Gutenberg
texte = re.sub(r"[ \t]+", " ", texte)                # espaces multiples → 1
texte = re.sub(r" ?\n ?", "\n", texte)               # espaces autour des retours ligne
texte = re.sub(r"\n{2,}", "\n", texte)               # lignes vides → 1 retour ligne (= fin de paragraphe)
texte = texte.strip()
noter("normaliser espaces, retours ligne et soulignés", avant, len(texte), "bruit de mise en page, sans valeur pour le style")

frequences = collections.Counter(texte)
rares = [c for c, n in frequences.items() if n < 20]
avant = len(texte)
texte = "".join(c for c in texte if c not in rares)
noter(f"retirer {len(rares)} caractères rares (< 20 occurrences)", avant, len(texte), "vocabulaire plus petit pour le LSTM, perte négligeable")
print("Caractères rares retirés :", "".join(sorted(rares)))

LIMITE = 100_000 if MODE_RAPIDE else len(texte)
avant = len(texte)
texte = texte[:LIMITE]
noter(f"couper à {LIMITE} caractères (MODE_RAPIDE={MODE_RAPIDE})", avant, len(texte), "temps d'entraînement sur CPU")

coupure = int(len(texte) * 0.95)
texte_train, texte_test = texte[:coupure], texte[coupure:]
noter("mettre les 5 % finaux à part (texte_test)", len(texte), len(texte_train), "mesurer la perplexité sur du texte jamais vu")
pd.DataFrame(JOURNAL)

**Feature engineering pour du texte = choisir l'unité.** Un modèle ne lit pas des lettres mais des entiers. Trois découpages possibles : le **caractère** (petit vocabulaire, séquences longues — notre LSTM), le **mot** (vocabulaire énorme, mots inconnus), le **token** sous-mot (le compromis des LLM : le tokenizer de Qwen découpe « Dantès » en 2-3 morceaux). On construit les trois pour les comparer.

In [ ]:
caracteres = sorted(set(texte))
car_vers_id = {c: i for i, c in enumerate(caracteres)}
id_vers_car = {i: c for c, i in car_vers_id.items()}
ids = np.array([car_vers_id[c] for c in texte_train], dtype=np.int32)

mots = re.findall(r"\w+|[^\w\s]", texte, flags=re.UNICODE)

from transformers import AutoTokenizer
NOM_LLM = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(NOM_LLM)
tokens_llm = tokenizer(texte_train[:20_000], add_special_tokens=False)["input_ids"]

tableau_unites = pd.DataFrame({
    "unité": ["caractère", "mot", "token Qwen (sur 20 000 car.)"],
    "taille du vocabulaire": [len(caracteres), len(set(mots)), tokenizer.vocab_size],
    "longueur de la séquence": [len(ids), len(mots), len(tokens_llm)],
    "caractères par unité": [1, round(len(texte) / len(mots), 2), round(20_000 / len(tokens_llm), 2)],
})
print("Vocabulaire :", "".join(caracteres))
print("« Dantès » en tokens Qwen :", [tokenizer.decode([t]) for t in tokenizer("Dantès", add_special_tokens=False)["input_ids"]])
tableau_unites

**À toi 1 · Découper en phrases**

Découpe `texte` en **phrases** avec `re.split` : une phrase se termine par `.`, `!`, `?` ou `…` suivi d'un espace ou d'un retour ligne. Range le résultat dans `phrases` (liste de chaînes non vides, sans espaces au bord). Exemple attendu : `phrases[0]` commence par le début du roman.

<details><summary>Indice</summary>

Un *lookbehind* `(?<=[.!?…])` permet de couper *après* la ponctuation sans la perdre : `re.split(r'(?<=[.!?…])\s+', texte)`.
</details>


In [ ]:
# À toi : découpe texte en phrases
phrases = [texte]      # ← remplace par le vrai découpage

phrases = [p.strip() for p in phrases if p.strip()]
verifier("Exercice 1 · au moins 20 phrases", len(phrases) >= 20)
verifier("Exercice 1 · aucune phrase vide", all(len(p) > 0 for p in phrases))
print(len(phrases), "phrases · exemple :", phrases[min(3, len(phrases) - 1)][:120])

<details><summary>Solution</summary>

```python
phrases = re.split(r"(?<=[.!?…])[\s\n]+", texte)

phrases = [p.strip() for p in phrases if p.strip()]
verifier("Exercice 1 · au moins 20 phrases", len(phrases) >= 20)
verifier("Exercice 1 · aucune phrase vide", all(len(p) > 0 for p in phrases))
print(len(phrases), "phrases · exemple :", phrases[min(3, len(phrases) - 1)][:120])
```
</details>


**À toi 2 · Encoder et décoder**

Écris `encoder(chaine)` qui renvoie la liste des ids (via `car_vers_id`) et `decoder(liste_ids)` qui refait la chaîne (via `id_vers_car`). Un caractère absent du vocabulaire est ignoré. Le test : un aller-retour sur les 60 premiers caractères du corpus doit rendre exactement le texte de départ.


In [ ]:
# À toi : encoder / décoder
def encoder(chaine):
    return []                                   # ← liste d'entiers

def decoder(liste_ids):
    return ""                                   # ← chaîne

extrait = texte[:60]
verifier("Exercice 2 · aller-retour exact", decoder(encoder(extrait)) == extrait)
verifier("Exercice 2 · caractère inconnu ignoré", decoder(encoder("a☃b")) == "ab")
print(encoder(extrait)[:12], "→", decoder(encoder(extrait))[:30])

<details><summary>Solution</summary>

```python
def encoder(chaine):
    return [car_vers_id[c] for c in chaine if c in car_vers_id]

def decoder(liste_ids):
    return "".join(id_vers_car[i] for i in liste_ids)

extrait = texte[:60]
verifier("Exercice 2 · aller-retour exact", decoder(encoder(extrait)) == extrait)
verifier("Exercice 2 · caractère inconnu ignoré", decoder(encoder("a☃b")) == "ab")
print(encoder(extrait)[:12], "→", decoder(encoder(extrait))[:30])
```
</details>


In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
if decoder(encoder("test")) != "test":
    def encoder(chaine):
        return [car_vers_id[c] for c in chaine if c in car_vers_id]

    def decoder(liste_ids):
        return "".join(id_vers_car[i] for i in liste_ids)


## 4. Analyse exploratoire

Quatre indicateurs pour connaître le corpus avant de modéliser : quels caractères, quelles longueurs, quels mots, et la loi de Zipf. Puis les « corrélations » d'un texte : quel caractère suit quel caractère.

In [ ]:
freq = pd.Series(collections.Counter(texte)).sort_values(ascending=False)
etiquettes = [repr(c) if c in "\n " else c for c in freq.index[:25]]
plt.figure(figsize=(10, 3.2))
plt.bar(etiquettes, freq.values[:25] / len(texte) * 100)
plt.ylabel("% du texte"); plt.title("Indicateur 1 · les 25 caractères les plus fréquents"); plt.tight_layout(); plt.show()
print(f"L'espace et le « e » font à eux deux {freq[[' ', 'e']].sum() / len(texte) * 100:.0f} % du texte.")

In [ ]:
longueurs_mots = [len(m) for m in mots if m.isalpha()]
longueurs_phrases = [len(p.split()) for p in re.split(r"(?<=[.!?…])\s+", texte) if p.strip()]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].hist(longueurs_mots, bins=range(1, 20), edgecolor="white"); axes[0].set_title("Indicateur 2 · longueur des mots (lettres)")
axes[1].hist(longueurs_phrases, bins=range(0, 80, 3), edgecolor="white"); axes[1].set_title("longueur des phrases (mots)")
plt.tight_layout(); plt.show()
print(f"Mot moyen : {np.mean(longueurs_mots):.1f} lettres · phrase moyenne : {np.mean(longueurs_phrases):.1f} mots · médiane {np.median(longueurs_phrases):.0f}")

*Lecture :* indicateur 1, l'espace et « e » dominent comme dans tout texte français : un modèle qui n'apprend que ces fréquences (la baseline « bête ») aura déjà une perplexité bien inférieure au hasard. Indicateur 2, des phrases courtes (les dialogues) et une longue traîne de phrases de 40 mots et plus : le modèle devra tenir une dépendance sur des dizaines de caractères — hors de portée d'un bigramme.

**À toi 3 · Les mots qui font le style**

Calcule les **15 mots les plus fréquents hors mots vides** (liste `MOTS_VIDES` fournie, comparaison en minuscules, mots d'au moins 3 lettres) dans `top_mots` (un `pd.Series` mot → effectif, trié décroissant) et trace-les en barres horizontales. Résultat attendu avec Dumas : des noms de personnages (« dantès », « morrel ») et des marqueurs du récit (« dit », « monsieur »).

<details><summary>Indice</summary>

Filtre dans la compréhension : `if m.isalpha() and len(m) >= 3 and m.lower() not in MOTS_VIDES`.
</details>


In [ ]:
MOTS_VIDES = set("le la les de des du un une et à en que qui ne pas ce se sa son ses il elle ils on nous vous je tu me te lui leur au aux y est était avait pour par sur dans avec plus mais comme cette ces tout tous fut".split())

# À toi : top_mots
compteur = collections.Counter(m.lower() for m in mots if m.isalpha())
top_mots = pd.Series(compteur).sort_values(ascending=False).head(15)     # ← filtre les mots vides et les mots < 3 lettres

verifier("Exercice 3 · 15 mots", len(top_mots) == 15)
verifier("Exercice 3 · aucun mot vide", not any(m in MOTS_VIDES for m in top_mots.index))
verifier("Exercice 3 · mots d'au moins 3 lettres", all(len(m) >= 3 for m in top_mots.index))
top_mots.sort_values().plot.barh(figsize=(6, 4), title="Indicateur 3 · mots les plus fréquents (hors mots vides)"); plt.tight_layout(); plt.show()

<details><summary>Solution</summary>

```python
MOTS_VIDES = set("le la les de des du un une et à en que qui ne pas ce se sa son ses il elle ils on nous vous je tu me te lui leur au aux y est était avait pour par sur dans avec plus mais comme cette ces tout tous fut".split())

compteur = collections.Counter(m.lower() for m in mots if m.isalpha() and len(m) >= 3 and m.lower() not in MOTS_VIDES)
top_mots = pd.Series(compteur).sort_values(ascending=False).head(15)

verifier("Exercice 3 · 15 mots", len(top_mots) == 15)
verifier("Exercice 3 · aucun mot vide", not any(m in MOTS_VIDES for m in top_mots.index))
verifier("Exercice 3 · mots d'au moins 3 lettres", all(len(m) >= 3 for m in top_mots.index))
top_mots.sort_values().plot.barh(figsize=(6, 4), title="Indicateur 3 · mots les plus fréquents (hors mots vides)"); plt.tight_layout(); plt.show()
```
</details>


In [ ]:
effectifs = np.sort(np.array(list(collections.Counter(m.lower() for m in mots if m.isalpha()).values())))[::-1]
rangs = np.arange(1, len(effectifs) + 1)
plt.figure(figsize=(5, 3.2))
plt.loglog(rangs, effectifs, ".", markersize=3)
plt.loglog(rangs, effectifs[0] / rangs, "--", label="loi de Zipf : f ∝ 1/rang")
plt.xlabel("rang du mot"); plt.ylabel("effectif"); plt.title("Indicateur 4 · loi de Zipf"); plt.legend(); plt.tight_layout(); plt.show()
couverture = effectifs[:100].sum() / effectifs.sum() * 100
print(f"Les 100 mots les plus fréquents couvrent {couverture:.0f} % du texte ; {np.sum(effectifs == 1)} mots n'apparaissent qu'une fois.")

*Lecture :* la loi de Zipf tient (droite en log-log) : une poignée de mots fait l'essentiel du texte, et des milliers de mots n'apparaissent qu'une fois. C'est ce qui rend le découpage en mots fragile, et le découpage en caractères ou en sous-mots plus robuste.

**Corrélations d'un texte : la matrice de transition.** À la place d'une matrice de corrélation, on regarde la probabilité qu'un caractère `b` suive un caractère `a`. C'est exactement le **modèle bigramme** de la séance 9, et ce sera notre première baseline.

In [ ]:
V = len(caracteres)
comptes = np.ones((V, V))                     # lissage +1 : aucune transition n'a une probabilité nulle
for a, b in zip(ids[:-1], ids[1:]):
    comptes[a, b] += 1
P_bigramme = comptes / comptes.sum(axis=1, keepdims=True)

lettres = [c for c in "abcdefghijklmnopqrstuvwxyz" if c in car_vers_id]
idx = [car_vers_id[c] for c in lettres]
plt.figure(figsize=(7, 6))
plt.imshow(P_bigramme[np.ix_(idx, idx)], cmap="Blues")
plt.xticks(range(len(lettres)), lettres); plt.yticks(range(len(lettres)), lettres)
plt.xlabel("caractère suivant"); plt.ylabel("caractère courant"); plt.title("P(suivant | courant) sur les lettres"); plt.colorbar(); plt.tight_layout(); plt.show()
q = car_vers_id.get("q"); u = car_vers_id.get("u")
if q is not None and u is not None:
    print(f"Après un « q », probabilité d'un « u » : {P_bigramme[q, u]:.2f}")

In [ ]:
print("=== Rapport · section 4 (à recopier dans le gabarit) ===")
print("Corpus :", SOURCE)
print("Caractères :", len(texte), "· vocabulaire :", V, "caractères · mots :", len(mots), "· mots distincts :", len(set(m.lower() for m in mots)))
print(f"Phrase moyenne : {np.mean(longueurs_phrases):.1f} mots · couverture des 100 mots les plus fréquents : {couverture:.0f} %")
print("Extrait test tenu à part :", len(texte_test), "caractères")

## 5. Modèles candidats

**Ce que l'exploration nous a dit.** Les régularités locales sont fortes (q → u, espace après ponctuation) : un bigramme les capte. Mais le style de Dumas tient à des dépendances longues (un nom propre de 6 lettres, un accord sujet-verbe, un dialogue ouvert par un tiret) : il faut une mémoire (LSTM) ou, mieux, un modèle qui connaît déjà le français et qu'on spécialise (LLM + LoRA).

Quatre candidats, du plus bête au plus gros, tous évalués de la même façon : **perplexité par caractère sur `texte_test`** (jamais vu) et génération sur les **mêmes amorces**.

| modèle | ce qu'il apprend | paramètres |
|---|---|---|
| unigramme (baseline « bête ») | la fréquence de chaque caractère | ≈ 90 |
| bigramme (séance 9) | P(caractère suivant \| caractère courant) | ≈ 8 000 |
| LSTM caractère (Keras) | une mémoire de 100 caractères | ≈ 350 000 |
| Qwen2.5-0.5B-Instruct (sans adaptation) | le français, tout Internet | 494 000 000 |

Perplexité par caractère = `exp(− moyenne des log P(caractère | contexte))`. Pour Qwen, on somme les log-probabilités des tokens puis on divise par le **nombre de caractères**, pour que les chiffres soient comparables.

**Validation propre.** Pas de validation croisée ici (on ne mélange jamais un texte : l'ordre est l'information), mais un **hold-out** : les 5 % finaux du corpus n'ont servi à aucun modèle, et la perplexité y est calculée avec la même fonction pour tous. La colonne « temps » compte l'entraînement seul (Qwen sans adaptation : 0, mais un pré-entraînement de plusieurs semaines de GPU en amont).

In [ ]:
AMORCES = ["Le 24 février 1815, la vigie de Notre-Dame de la Garde",
           "Dantès regarda le vieillard et",
           "— Monsieur, dit"]
if SOURCE == "mon texte":
    AMORCES = [texte[:50], texte[1000:1030], texte[2000:2020]]     # amorces tirées de ton corpus
GENERATIONS = {}          # modèle → liste de textes générés (une par amorce)
RESULTATS = []            # lignes du tableau comparatif

def noter_resultat(modele, perplexite, parametres, temps):
    RESULTATS.append({"modèle": modele, "perplexité / caractère": round(perplexite, 2),
                      "paramètres": parametres, "temps d'entraînement (s)": round(temps, 1)})
    return pd.DataFrame(RESULTATS)

ids_test = np.array([car_vers_id[c] for c in texte_test])
P_uni = np.bincount(ids, minlength=V) + 1.0
P_uni = P_uni / P_uni.sum()

ppl_uni = math.exp(-np.mean(np.log(P_uni[ids_test])))
ppl_bi = math.exp(-np.mean(np.log(P_bigramme[ids_test[:-1], ids_test[1:]])))
print(f"Hasard uniforme : {V} · unigramme : {ppl_uni:.1f} · bigramme : {ppl_bi:.1f}")
noter_resultat("unigramme", ppl_uni, V, 0)
noter_resultat("bigramme", ppl_bi, V * V, 0)

**À toi 4 · Générer avec le bigramme**

Écris `generer_bigramme(amorce, n)` : à partir du dernier caractère de l'amorce, tire le caractère suivant selon `P_bigramme[courant]` (`np.random.choice`), ajoute-le, et recommence `n` fois. Renvoie l'amorce + les `n` caractères générés. Attendu : du charabia qui *ressemble* à du français (voyelles et consonnes alternées, espaces raisonnables).


In [ ]:
# À toi : génération bigramme
def generer_bigramme(amorce, n=150):
    resultat = amorce
    # ← boucle : courant = car_vers_id[resultat[-1]] ; suivant = np.random.choice(V, p=P_bigramme[courant]) ; ajoute id_vers_car[suivant]
    return resultat

np.random.seed(GRAINE)
GENERATIONS["bigramme"] = [generer_bigramme(a) for a in AMORCES]
verifier("Exercice 4 · longueur amorce + 150", all(len(g) == len(a) + 150 for g, a in zip(GENERATIONS["bigramme"], AMORCES)))
verifier("Exercice 4 · caractères du vocabulaire", all(c in car_vers_id for g in GENERATIONS["bigramme"] for c in g))
print(GENERATIONS["bigramme"][0])

<details><summary>Solution</summary>

```python
def generer_bigramme(amorce, n=150):
    resultat = amorce
    for _ in range(n):
        courant = car_vers_id[resultat[-1]]
        suivant = np.random.choice(V, p=P_bigramme[courant])
        resultat += id_vers_car[suivant]
    return resultat

np.random.seed(GRAINE)
GENERATIONS["bigramme"] = [generer_bigramme(a) for a in AMORCES]
verifier("Exercice 4 · longueur amorce + 150", all(len(g) == len(a) + 150 for g, a in zip(GENERATIONS["bigramme"], AMORCES)))
verifier("Exercice 4 · caractères du vocabulaire", all(c in car_vers_id for g in GENERATIONS["bigramme"] for c in g))
print(GENERATIONS["bigramme"][0])
```
</details>


In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
if len(GENERATIONS.get("bigramme", [""])[0]) < len(AMORCES[0]) + 150:
    def generer_bigramme(amorce, n=150):
        resultat = amorce
        for _ in range(n):
            suivant = np.random.choice(V, p=P_bigramme[car_vers_id[resultat[-1]]])
            resultat += id_vers_car[suivant]
        return resultat

    np.random.seed(GRAINE)
    GENERATIONS["bigramme"] = [generer_bigramme(a) for a in AMORCES]


### LSTM caractère par caractère (Keras)

Le principe de la brique Le Wagon / TensorFlow « text generation » : le modèle lit une fenêtre de `LONGUEUR` caractères et prédit, **à chaque position**, le caractère suivant (`return_sequences=True`, donc 100 cibles par fenêtre : un apprentissage bien plus riche que de prédire seulement le 101e). Les fenêtres se chevauchent (`PAS`) pour multiplier les exemples.

In [ ]:
LONGUEUR = 100
PAS = 25 if MODE_RAPIDE else 50
EPOCHS_LSTM = 8 if MODE_RAPIDE else 30

def fenetres(ids_serie, longueur, pas):
    debuts = range(0, len(ids_serie) - longueur - 1, pas)
    X = np.array([ids_serie[d:d + longueur] for d in debuts])
    Y = np.array([ids_serie[d + 1:d + longueur + 1] for d in debuts])
    return X, Y

X_lstm, Y_lstm = fenetres(ids, LONGUEUR, PAS)
X_lstm_test, Y_lstm_test = fenetres(ids_test, LONGUEUR, LONGUEUR)
print("Fenêtres d'entraînement :", X_lstm.shape, "· test :", X_lstm_test.shape)
print("Entrée :", repr(decoder(X_lstm[0][:40])), "\nCible  :", repr(decoder(Y_lstm[0][:40])))

In [ ]:
lstm = tf.keras.Sequential([
    tf.keras.layers.Embedding(V, 64),
    tf.keras.layers.LSTM(256, return_sequences=True),
    tf.keras.layers.Dense(V),                                     # logits, un par caractère du vocabulaire
])
lstm.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
lstm.summary()

debut = time.time()
histo = lstm.fit(X_lstm, Y_lstm, epochs=EPOCHS_LSTM, batch_size=64, validation_data=(X_lstm_test, Y_lstm_test), verbose=1)
TEMPS["LSTM"] = time.time() - debut

plt.figure(figsize=(5, 3))
plt.plot(np.exp(histo.history["loss"]), label="entraînement"); plt.plot(np.exp(histo.history["val_loss"]), label="test")
plt.xlabel("epoch"); plt.ylabel("perplexité / caractère"); plt.title("LSTM : la perplexité descend"); plt.legend(); plt.tight_layout(); plt.show()
print(f"Entraînement : {TEMPS['LSTM']:.0f} s")

**À toi 5 · La température**

Le modèle sort des logits ; on les divise par `T` avant le softmax. `T` petit (0,3) → on prend presque toujours le caractère le plus probable (texte sûr mais répétitif) ; `T` grand (1,5) → plus de surprises, plus de fautes.

Écris `appliquer_temperature(logits, T)` qui renvoie les probabilités `softmax(logits / T)` (un tableau qui somme à 1). Vérifie qu'avec `T = 0.5` la probabilité du caractère le plus probable **augmente** par rapport à `T = 1`, et qu'avec `T = 2` elle **diminue**.

<details><summary>Indice</summary>

Soustraire le max avant `np.exp` évite les dépassements numériques : `p = np.exp(z - z.max()); p / p.sum()`.
</details>


In [ ]:
# À toi : softmax avec température
def appliquer_temperature(logits, T=1.0):
    p = np.ones(len(logits)) / len(logits)      # ← remplace : exp((logits/T) − max) puis normalise
    return p

logits_exemple = np.array([2.0, 1.0, 0.5, -1.0])
p1, p_froid, p_chaud = (appliquer_temperature(logits_exemple, T) for T in (1.0, 0.5, 2.0))
verifier("Exercice 5 · somme à 1", proche(p1.sum(), 1.0, 1e-6))
verifier("Exercice 5 · T=0.5 rend le max plus probable", p_froid.max() > p1.max())
verifier("Exercice 5 · T=2 rend le max moins probable", p_chaud.max() < p1.max())
print("T=1 :", p1.round(3), "· T=0.5 :", p_froid.round(3), "· T=2 :", p_chaud.round(3))

<details><summary>Solution</summary>

```python
def appliquer_temperature(logits, T=1.0):
    z = np.asarray(logits, dtype=np.float64) / T
    p = np.exp(z - z.max())
    return p / p.sum()

logits_exemple = np.array([2.0, 1.0, 0.5, -1.0])
p1, p_froid, p_chaud = (appliquer_temperature(logits_exemple, T) for T in (1.0, 0.5, 2.0))
verifier("Exercice 5 · somme à 1", proche(p1.sum(), 1.0, 1e-6))
verifier("Exercice 5 · T=0.5 rend le max plus probable", p_froid.max() > p1.max())
verifier("Exercice 5 · T=2 rend le max moins probable", p_chaud.max() < p1.max())
print("T=1 :", p1.round(3), "· T=0.5 :", p_froid.round(3), "· T=2 :", p_chaud.round(3))
```
</details>


In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
_l = np.array([2.0, 1.0, 0.5, -1.0])
if abs(appliquer_temperature(_l, 0.5).max() - appliquer_temperature(_l, 2.0).max()) < 1e-9:
    def appliquer_temperature(logits, T=1.0):
        z = np.asarray(logits, dtype=np.float64) / T
        p = np.exp(z - z.max())
        return p / p.sum()


In [ ]:
def generer_lstm(amorce, n=150, T=0.7):
    resultat = amorce
    for _ in range(n):
        x = np.array([encoder(resultat[-LONGUEUR:])])
        logits = lstm(x, training=False).numpy()[0, -1]
        p = appliquer_temperature(logits, T)
        resultat += id_vers_car[np.random.choice(V, p=p)]
    return resultat

np.random.seed(GRAINE)
GENERATIONS["LSTM"] = [generer_lstm(a) for a in AMORCES]
for g in GENERATIONS["LSTM"]:
    print(g.replace("\n", " ⏎ "), "\n---")

ppl_lstm = math.exp(lstm.evaluate(X_lstm_test, Y_lstm_test, verbose=0))
noter_resultat("LSTM caractère", ppl_lstm, lstm.count_params(), TEMPS["LSTM"])

### Qwen2.5-0.5B-Instruct, sans adaptation

Un LLM de 494 millions de paramètres, entraîné par Alibaba sur des milliers de milliards de tokens, multilingue, téléchargé depuis Hugging Face (≈ 1 Go, mis en cache). On lui donne l'amorce brute (pas de format « chat ») et on le laisse continuer. Sa perplexité est mesurée par fenêtres de 512 tokens, puis ramenée **par caractère**.

In [ ]:
from transformers import AutoModelForCausalLM

debut = time.time()
llm = AutoModelForCausalLM.from_pretrained(NOM_LLM, dtype=torch.float32).to(APPAREIL)
print(f"Chargé en {time.time() - debut:.0f} s ·", f"{sum(p.numel() for p in llm.parameters()) / 1e6:.0f} M paramètres")

def generer_llm(modele, amorce, n=60, T=0.7, top_p=0.9):
    modele.eval()
    entree = tokenizer(amorce, return_tensors="pt").to(APPAREIL)
    with torch.no_grad():
        sortie = modele.generate(**entree, max_new_tokens=n, do_sample=True, temperature=T, top_p=top_p,
                                 pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(sortie[0], skip_special_tokens=True)

In [ ]:
@torch.no_grad()
def perplexite_llm(modele, texte_eval, fenetre=512):
    """Perplexité PAR CARACTÈRE : somme des log-pertes sur les tokens / nombre de caractères."""
    modele.eval()
    tokens = tokenizer(texte_eval, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    perte_totale = 0.0
    for d in range(0, len(tokens) - 1, fenetre):
        x = tokens[d:d + fenetre + 1][None, :].to(APPAREIL)
        perte_totale += modele(input_ids=x, labels=x).loss.item() * (x.shape[1] - 1)
    return math.exp(perte_totale / len(texte_eval))

debut = time.time()
ppl_qwen = perplexite_llm(llm, texte_test)
print(f"Perplexité / caractère de Qwen sur texte_test : {ppl_qwen:.2f}  ({time.time() - debut:.0f} s)")

In [ ]:
torch.manual_seed(GRAINE)
GENERATIONS["Qwen"] = [generer_llm(llm, a) for a in AMORCES]
for g in GENERATIONS["Qwen"]:
    print(g.replace("\n", " ⏎ "), "\n---")
noter_resultat("Qwen2.5-0.5B (sans adaptation)", ppl_qwen, 494_000_000, 0)

In [ ]:
print("=== Rapport · section 5 (à recopier dans le gabarit) ===")
print(pd.DataFrame(RESULTATS).to_string(index=False))
print("\nAmorce :", AMORCES[1])
for modele in GENERATIONS:
    print(f"[{modele}]", GENERATIONS[modele][1][len(AMORCES[1]):len(AMORCES[1]) + 120].replace("\n", " ⏎ "))

## 6. Fine-tuning : Qwen + LoRA

Qwen écrit un français correct mais générique. On veut le **spécialiser** sur Dumas sans toucher à ses 494 millions de paramètres (trop lourd, et on risquerait de lui faire oublier le français). **LoRA** (*Low-Rank Adaptation*, Hu et al. 2021) ajoute à côté de certaines matrices d'attention deux petites matrices `A` (d × r) et `B` (r × d) : seule leur multiplication `B·A` est apprise. Avec un rang `r = 8` sur les projections `q_proj` et `v_proj` des 24 couches, on entraîne **0,1 % des paramètres**. C'est la méthode standard pour adapter un LLM sur un seul GPU (ou, en version miniature, sur un CPU).

Le corpus est découpé en exemples de 128 tokens ; le modèle apprend à prédire chaque token à partir des précédents (même tâche qu'en pré-entraînement, `DataCollatorForLanguageModeling(mlm=False)`).

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

LONG_TOKENS = 128
tokens_train = tokenizer(texte_train, add_special_tokens=False)["input_ids"]
exemples = [{"input_ids": tokens_train[d:d + LONG_TOKENS], "attention_mask": [1] * LONG_TOKENS}
            for d in range(0, len(tokens_train) - LONG_TOKENS, LONG_TOKENS)]
N_EXEMPLES = 64 if MODE_RAPIDE else len(exemples)
EPOCHS_LORA = 1 if MODE_RAPIDE else 3
exemples = exemples[:N_EXEMPLES]
print(len(tokens_train), "tokens →", len(exemples), "exemples de", LONG_TOKENS, "tokens ·", EPOCHS_LORA, "epoch(s)")

config_lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
llm_lora = get_peft_model(llm, config_lora)
llm_lora.print_trainable_parameters()
nb_entrainables, nb_total = llm_lora.get_nb_trainable_parameters()

**À toi 6 · Compter les paramètres LoRA**

Retrouve par le calcul le nombre de paramètres entraînables affiché ci-dessus. Pour chaque couche (`llm.config.num_hidden_layers`) et chaque module ciblé, LoRA ajoute `r × (dim_entrée + dim_sortie)` paramètres. Dimensions : `q_proj` va de `hidden_size` vers `hidden_size` ; `v_proj` va de `hidden_size` vers `num_key_value_heads × (hidden_size / num_attention_heads)`. Calcule `params_r8`, puis `params_r16` (même formule avec r = 16).

<details><summary>Indice</summary>

`params = num_hidden_layers × (r × (dim_q[0] + dim_q[1]) + r × (dim_v[0] + dim_v[1]))`.
</details>


In [ ]:
cfg = llm.config
r = 8
dim_q = (cfg.hidden_size, cfg.hidden_size)
dim_v = (cfg.hidden_size, cfg.num_key_value_heads * cfg.hidden_size // cfg.num_attention_heads)

# À toi : params_r8 et params_r16
params_r8 = None
params_r16 = None

verifier("Exercice 6 · r=8 retrouvé", params_r8 == nb_entrainables)
verifier("Exercice 6 · r=16 = le double", params_r16 == 2 * nb_entrainables)
print("Calculé :", params_r8, "· mesuré :", nb_entrainables, f"· soit {nb_entrainables / nb_total * 100:.2f} % du modèle")

<details><summary>Solution</summary>

```python
cfg = llm.config
r = 8
dim_q = (cfg.hidden_size, cfg.hidden_size)
dim_v = (cfg.hidden_size, cfg.num_key_value_heads * cfg.hidden_size // cfg.num_attention_heads)

params_r8 = cfg.num_hidden_layers * (r * sum(dim_q) + r * sum(dim_v))
params_r16 = cfg.num_hidden_layers * (16 * sum(dim_q) + 16 * sum(dim_v))

verifier("Exercice 6 · r=8 retrouvé", params_r8 == nb_entrainables)
verifier("Exercice 6 · r=16 = le double", params_r16 == 2 * nb_entrainables)
print("Calculé :", params_r8, "· mesuré :", nb_entrainables, f"· soit {nb_entrainables / nb_total * 100:.2f} % du modèle")
```
</details>


In [ ]:
arguments = TrainingArguments(
    output_dir="data/lora_dumas", per_device_train_batch_size=8, num_train_epochs=EPOCHS_LORA,
    learning_rate=2e-4, logging_steps=2, save_strategy="no", report_to="none",
    fp16=torch.cuda.is_available(), use_cpu=not torch.cuda.is_available(), dataloader_pin_memory=False, seed=GRAINE,
)
entraineur = Trainer(model=llm_lora, args=arguments, train_dataset=exemples,
                     data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
debut = time.time()
entraineur.train()
TEMPS["Qwen + LoRA"] = time.time() - debut
print(f"Fine-tuning LoRA : {TEMPS['Qwen + LoRA']:.0f} s")

In [ ]:
journal_lora = pd.DataFrame([l for l in entraineur.state.log_history if "loss" in l])
plt.figure(figsize=(5, 3))
plt.plot(journal_lora["step"], journal_lora["loss"], marker="o")
plt.xlabel("pas d'entraînement"); plt.ylabel("perte (log-perplexité / token)"); plt.title("LoRA : courbe d'apprentissage"); plt.tight_layout(); plt.show()

ppl_lora = perplexite_llm(llm_lora, texte_test)
print(f"Perplexité / caractère · avant LoRA : {ppl_qwen:.2f} · après LoRA : {ppl_lora:.2f}")
noter_resultat("Qwen2.5-0.5B + LoRA", ppl_lora, nb_entrainables, TEMPS["Qwen + LoRA"])

**Avant / après sur les mêmes amorces.** En mode rapide (64 exemples, 1 epoch), le changement de style est discret ; en mode complet (tout le tome, 3 epochs sur T4) le modèle prend l'accent de Dumas : tirets de dialogue, « Monsieur », vocabulaire de marine.

In [ ]:
torch.manual_seed(GRAINE)
GENERATIONS["Qwen + LoRA"] = [generer_llm(llm_lora, a) for a in AMORCES]
for amorce, avant, apres in zip(AMORCES, GENERATIONS["Qwen"], GENERATIONS["Qwen + LoRA"]):
    print("AMORCE :", amorce)
    print("  avant :", avant[len(amorce):].replace("\n", " ⏎ "))
    print("  après :", apres[len(amorce):].replace("\n", " ⏎ "), "\n")

**À toi 7 · Ta propre amorce, tes réglages**

`top_p = 0.9` (*nucleus sampling*) ne tire que parmi les tokens dont les probabilités cumulées atteignent 90 % : on coupe la longue traîne des tokens absurdes. Combiné à la température, c'est le réglage que tu retrouves dans toutes les API de LLM.

Choisis une amorce à toi (`MON_AMORCE`) et génère avec le modèle adapté pour trois réglages : `(T=0.3, top_p=0.9)`, `(T=0.8, top_p=0.9)`, `(T=1.3, top_p=1.0)`. Note dans `OBSERVATION` en une phrase ce qui change (répétitions ? fautes ? cohérence ?).

Check-list : ☐ trois sorties affichées · ☐ la sortie à T=0.3 est la plus prévisible · ☐ `OBSERVATION` remplie.


In [ ]:
# À toi : amorce et réglages
MON_AMORCE = "Le vieux marin"
REGLAGES = [(0.3, 0.9), (0.8, 0.9), (1.3, 1.0)]
OBSERVATION = ""                     # ← ta phrase

torch.manual_seed(GRAINE)
for T, top_p in REGLAGES:
    print(f"T={T} top_p={top_p} →", generer_llm(llm_lora, MON_AMORCE, n=50, T=T, top_p=top_p).replace("\n", " ⏎ "), "\n")
verifier("Exercice 7 · observation notée", len(OBSERVATION) > 10)

<details><summary>Solution</summary>

```python
MON_AMORCE = "Le vieux marin"
REGLAGES = [(0.3, 0.9), (0.8, 0.9), (1.3, 1.0)]
OBSERVATION = "À T=0.3 le texte est propre mais répète des tournures ; à T=1.3 sans top-p il invente des mots et perd le fil."

torch.manual_seed(GRAINE)
for T, top_p in REGLAGES:
    print(f"T={T} top_p={top_p} →", generer_llm(llm_lora, MON_AMORCE, n=50, T=T, top_p=top_p).replace("\n", " ⏎ "), "\n")
verifier("Exercice 7 · observation notée", len(OBSERVATION) > 10)
```
</details>


In [ ]:
print("=== Rapport · section 6 (à recopier dans le gabarit) ===")
print(pd.DataFrame(RESULTATS).to_string(index=False))
print(f"\nLoRA : r=8 sur q_proj/v_proj · {nb_entrainables:,} paramètres entraînés ({nb_entrainables / nb_total * 100:.2f} %) · "
      f"{len(exemples)} exemples × {LONG_TOKENS} tokens · {EPOCHS_LORA} epoch(s) · {TEMPS['Qwen + LoRA']:.0f} s sur {APPAREIL}")
print(f"Perplexité Qwen : {ppl_qwen:.2f} → {ppl_lora:.2f} après LoRA ({(1 - ppl_lora / ppl_qwen) * 100:+.1f} %)")

## 7. Interprétation

Trois questions concrètes : les modèles **recopient**-ils le livre (mémorisation) ? Que fait la **température** à la diversité ? Et qu'est-ce que le modèle « pense » à un instant donné (les probabilités du caractère suivant) ? Puis une grille de lecture qualitative à remplir.

In [ ]:
def taux_copie(texte_genere, corpus, n=8):
    """Part des n-grammes de caractères du texte généré qui existent tels quels dans le corpus d'entraînement."""
    grammes = [texte_genere[i:i + n] for i in range(len(texte_genere) - n + 1)]
    return sum(g in corpus for g in grammes) / max(len(grammes), 1)

copie = {m: np.mean([taux_copie(g[len(a):], texte_train) for g, a in zip(GENERATIONS[m], AMORCES)]) for m in GENERATIONS}
copie["texte_test (référence)"] = taux_copie(texte_test[:500], texte_train)
pd.Series(copie).plot.barh(figsize=(6, 3), title="Part des 8-grammes générés présents dans le corpus d'entraînement")
plt.xlabel("taux de copie"); plt.tight_layout(); plt.show()

**À toi 8 · Diversité selon la température**

*Lecture du graphique :* la référence (un vrai extrait de Dumas jamais vu) indique le taux « naturel » de 8-grammes communs (mots courants, tournures). Un modèle **au-dessus** de la référence recopie ; **très en dessous**, il invente des suites de lettres qui n'existent pas (le bigramme). Le bon modèle est proche de la référence.

Écris `diversite(texte_genere, n=3)` = nombre de 3-grammes **distincts** divisé par le nombre total de 3-grammes (1 = jamais de répétition). Génère 200 caractères avec le LSTM pour `T` dans `[0.3, 0.7, 1.0, 1.5]`, range les diversités dans `DIVERSITES` (dict T → valeur) et trace la courbe. Attendu : la diversité **monte** avec la température.


In [ ]:
# À toi : diversité vs température
def diversite(texte_genere, n=3):
    return 0.0                                    # ← nb de n-grammes distincts / nb total

DIVERSITES = {}
np.random.seed(GRAINE)
for T in [0.3, 0.7, 1.0, 1.5]:
    DIVERSITES[T] = diversite(generer_lstm(AMORCES[1], n=200, T=T)[len(AMORCES[1]):])

verifier("Exercice 8 · valeurs entre 0 et 1", all(0 <= v <= 1 for v in DIVERSITES.values()))
verifier("Exercice 8 · plus de diversité à T=1.5 qu'à T=0.3", DIVERSITES.get(1.5, 0) > DIVERSITES.get(0.3, 1))
plt.figure(figsize=(4.5, 3)); plt.plot(list(DIVERSITES), list(DIVERSITES.values()), marker="o")
plt.xlabel("température"); plt.ylabel("diversité des 3-grammes"); plt.tight_layout(); plt.show()

<details><summary>Solution</summary>

```python
def diversite(texte_genere, n=3):
    grammes = [texte_genere[i:i + n] for i in range(len(texte_genere) - n + 1)]
    return len(set(grammes)) / max(len(grammes), 1)

DIVERSITES = {}
np.random.seed(GRAINE)
for T in [0.3, 0.7, 1.0, 1.5]:
    DIVERSITES[T] = diversite(generer_lstm(AMORCES[1], n=200, T=T)[len(AMORCES[1]):])

verifier("Exercice 8 · valeurs entre 0 et 1", all(0 <= v <= 1 for v in DIVERSITES.values()))
verifier("Exercice 8 · plus de diversité à T=1.5 qu'à T=0.3", DIVERSITES.get(1.5, 0) > DIVERSITES.get(0.3, 1))
plt.figure(figsize=(4.5, 3)); plt.plot(list(DIVERSITES), list(DIVERSITES.values()), marker="o")
plt.xlabel("température"); plt.ylabel("diversité des 3-grammes"); plt.tight_layout(); plt.show()
```
</details>


In [ ]:
#@title 🛟 Filet de sécurité — déplie seulement si tu n'as pas fait l'exercice { display-mode: "form" }
# Réinjecte l'implémentation de référence pour que la suite du notebook fonctionne.
if max(DIVERSITES.values(), default=0) == 0:
    def diversite(texte_genere, n=3):
        grammes = [texte_genere[i:i + n] for i in range(len(texte_genere) - n + 1)]
        return len(set(grammes)) / max(len(grammes), 1)

    np.random.seed(GRAINE)
    DIVERSITES = {T: diversite(generer_lstm(AMORCES[1], n=200, T=T)[len(AMORCES[1]):]) for T in [0.3, 0.7, 1.0, 1.5]}


**Ce que le modèle « pense ».** Après une amorce, on affiche les 8 caractères les plus probables selon le LSTM et les 8 tokens les plus probables selon Qwen + LoRA. C'est l'équivalent, pour un modèle de langue, d'une importance de variables : on voit ce que le contexte a activé.

In [ ]:
contexte = AMORCES[1]
logits_lstm = lstm(np.array([encoder(contexte[-LONGUEUR:])]), training=False).numpy()[0, -1]
p_lstm = appliquer_temperature(logits_lstm, 1.0)
top = np.argsort(p_lstm)[::-1][:8]

with torch.no_grad():
    entree = tokenizer(contexte, return_tensors="pt").to(APPAREIL)
    p_llm = torch.softmax(llm_lora(**entree).logits[0, -1], dim=-1).cpu().numpy()
top_llm = np.argsort(p_llm)[::-1][:8]

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].bar([repr(id_vers_car[i]) for i in top], p_lstm[top]); axes[0].set_title(f"LSTM · après « …{contexte[-20:]} »")
axes[1].bar([repr(tokenizer.decode([i])) for i in top_llm], p_llm[top_llm]); axes[1].set_title("Qwen + LoRA · token suivant")
plt.tight_layout(); plt.show()

**À toi 9 · Remplir la grille**

La perplexité ne dit pas si le texte est *bon*. Relis les générations de la section 6 (amorce 2) et note chaque modèle de 0 à 2 sur quatre critères : français correct, cohérence sur 2-3 phrases, style Dumas (dialogues, vocabulaire), nouveauté (ne recopie pas). Remplace les `None` de `GRILLE` par tes notes (0, 1 ou 2). Le total par modèle s'affiche ; compare-le au classement par perplexité : sont-ils d'accord ?


In [ ]:
# À toi : notes de 0 à 2
GRILLE = pd.DataFrame({
    "français correct": [None, None, None, None],
    "cohérence":        [None, None, None, None],
    "style Dumas":      [None, None, None, None],
    "nouveauté":        [None, None, None, None],
}, index=["bigramme", "LSTM", "Qwen", "Qwen + LoRA"])

verifier("Exercice 9 · grille remplie", GRILLE.notna().all().all())
GRILLE["total"] = GRILLE.sum(axis=1, numeric_only=True)
GRILLE

<details><summary>Solution</summary>

```python
GRILLE = pd.DataFrame({
    "français correct": [0, 1, 2, 2],
    "cohérence":        [0, 0, 2, 2],
    "style Dumas":      [0, 1, 1, 2],
    "nouveauté":        [2, 2, 2, 1],
}, index=["bigramme", "LSTM", "Qwen", "Qwen + LoRA"])

verifier("Exercice 9 · grille remplie", GRILLE.notna().all().all())
GRILLE["total"] = GRILLE.sum(axis=1, numeric_only=True)
GRILLE
```
</details>


**Trois enseignements.**
1. **Le pré-entraînement fait tout** : Qwen, sans avoir lu une ligne de Dumas, a une perplexité bien plus basse que le LSTM entraîné sur le livre. Apprendre le français depuis zéro sur un seul roman est hors de portée d'un petit modèle.
2. **LoRA suffit pour le style** : 0,1 % des paramètres, quelques minutes, et la perplexité descend ; en mode complet, le modèle adopte les tics de Dumas. Le prix : un risque de **mémorisation** (taux de copie qui monte) si on insiste trop.
3. **La température est un vrai levier métier** : basse pour un texte sûr, haute pour brainstormer. Le réglage se fait avec un humain devant la sortie, pas avec une métrique.

In [ ]:
print("=== Rapport · section 7 (à recopier dans le gabarit) ===")
for m, v in copie.items():
    print(f"taux de copie {m:28s} {v:.2f}")
print("Diversité LSTM selon T :", {T: round(v, 2) for T, v in DIVERSITES.items()})
print("Classement perplexité :", " < ".join(pd.DataFrame(RESULTATS).sort_values("perplexité / caractère")["modèle"]))

## 8. Conclusion et recommandation

**Réponse à la question métier.** Pour un assistant « style Dumas », la recommandation est **Qwen2.5-0.5B + LoRA** : c'est le seul candidat qui produit un français cohérent *et* qui se laisse spécialiser pour quelques minutes de GPU (ou de CPU en miniature). Le LSTM maison est un excellent outil pédagogique (on voit le modèle apprendre l'orthographe lettre à lettre) mais il faudrait des jours d'entraînement et bien plus de texte pour approcher un LLM. Le bigramme sert de plancher : tout ce qui ne le bat pas nettement ne vaut rien.

**Chiffre clé** : la perplexité par caractère (tableau de la section 6) — l'écart entre le LSTM et Qwen, puis le gain apporté par LoRA.

**Limites.** (1) *Hallucination* : le modèle invente des faits, des lieux, des dates avec aplomb ; rien ne garantit la cohérence narrative au-delà de quelques phrases. (2) *Mémorisation* : plus on fine-tune sur un petit corpus, plus il recopie ; à surveiller avec le taux de copie. (3) *Droits* : Dumas est libre, mais fine-tuner sur un auteur vivant sans autorisation pose un problème juridique et éthique — et le modèle de base a ses propres conditions d'usage (licence Apache 2.0 pour Qwen2.5). (4) La perplexité mesure la prédictibilité, pas la qualité littéraire.

**Avec plus de temps.** Fine-tuner sur les six tomes (Gutenberg les a tous), essayer `r = 32` et toutes les projections (`k_proj`, `o_proj`), un modèle de 1,5 B, une évaluation en aveugle « vrai ou faux Dumas » avec des lecteurs, et un garde-fou anti-copie (rejeter une génération dont le taux de copie dépasse un seuil).

In [ ]:
# À compléter : ta synthèse en 3 à 5 lignes (question, méthode, chiffre clé, recommandation, limite)
MA_SYNTHESE = """
...
"""
print(MA_SYNTHESE)

## 9. Pour aller plus loin

- **Le tutoriel d'origine** (Shakespeare, GRU caractère par caractère) : https://www.tensorflow.org/text/tutorials/text_generation — et l'article fondateur de Karpathy, *The Unreasonable Effectiveness of RNNs* : https://karpathy.github.io/2015/05/21/rnn-effectiveness/
- **LoRA** : l'article https://arxiv.org/abs/2106.09685, la documentation `peft` https://huggingface.co/docs/peft/index et le guide LoRA https://huggingface.co/docs/peft/task_guides/lora_based_methods
- **Entraîner un modèle de langue causal** avec `Trainer` (cours Hugging Face, chapitre 7) : https://huggingface.co/learn/llm-course/chapter7/6 — et la perplexité expliquée : https://huggingface.co/docs/transformers/main/en/perplexity
- **Réglages de génération** (température, top-k, top-p, beam search) : https://huggingface.co/docs/transformers/main/en/generation_strategies
- **Écrire un GPT de zéro** en 300 lignes (nanoGPT, Karpathy) : https://github.com/karpathy/nanoGPT — la suite logique du LSTM caractère.
- **Le modèle** : https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct · **le corpus** : https://www.gutenberg.org/ebooks/17989 (licence : https://www.gutenberg.org/policy/license.html)